In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# 크롬 옵션 설정
options = Options()
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36")
options.add_argument("--headless")

# 드라이버 실행
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 사람인 검색 결과 페이지 접속
search_url = "https://www.saramin.co.kr/zf_user/search/recruit?search_area=main&search_done=y&search_optional_item=n&searchType=search&searchword=데이터분석&recruitPage=1&recruitSort=relation&recruitPageCount=40"
driver.get(search_url)
time.sleep(5)

# 결과 저장용 리스트
job_list = []

# 채용공고 페이지 전체 가져오기
job_cards = driver.find_elements(By.CSS_SELECTOR, "div.item_recruit")

for card in job_cards:
    try:
        # 제목 + 링크
        title_element = card.find_element(By.CSS_SELECTOR, "h2.job_tit > a[title]")
        job_title = title_element.get_attribute("title")
        job_link = title_element.get_attribute("href")

        # 회사명
        company_element = card.find_element(By.CSS_SELECTOR, "strong.corp_name > a.track_event.data_layer")
        company_name = company_element.text

        # 세부정보 (여러 span 합치기)
        condition_spans = card.find_elements(By.CSS_SELECTOR, "div.job_condition > span")
        details = ', '.join([span.text for span in condition_spans if span.text.strip()])

        # 데이터 저장
        job_list.append({
            "Site": "Saramin",
            "Col_Company": company_name,
            "Col_Recruit": job_title,
            "Col_detail": details,
            "Col_url": job_link
        })

    except Exception as e:
        print(f"[공고 누락] 오류 발생: {e}")
        continue

# 드라이버 종료
driver.quit()

# DataFrame 생성
saramin_df = pd.DataFrame(job_list)


#display(saramin_df)


In [2]:
saramin_df.to_csv("data_tmp/data_Saramin.csv", index=False, encoding="utf-8-sig")